In [0]:
#Load Models & Predictions

from pyspark.ml.classification import RandomForestClassificationModel, DecisionTreeClassificationModel, LogisticRegressionModel, GBTClassificationModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col, when

# Load prepared data
df = spark.read.parquet("/Volumes/workspace/default/museum/prepared/")
df = df.filter(col("kingdom").isin(["Animalia", "Protista", "Plantae", "Chromista"]))
train, val, test = df.randomSplit([0.7, 0.15, 0.15], seed=42)

# Load saved models
model_path = "/Volumes/workspace/default/museum/models/"
rf_model = RandomForestClassificationModel.load(model_path + "random_forest")
dt_model = DecisionTreeClassificationModel.load(model_path + "decision_tree")
lr_model = LogisticRegressionModel.load(model_path + "logistic_regression")
gbt_model = GBTClassificationModel.load(model_path + "gbt")

# Generate predictions
rf_preds = rf_model.transform(test)
dt_preds = dt_model.transform(test)
lr_preds = lr_model.transform(test)

# GBT needs binary labels
test_binary = test.withColumn("label", when(col("label") == 0.0, 1.0).otherwise(0.0))
gbt_preds = gbt_model.transform(test_binary)

print("Models loaded and predictions generated successfully")
print(f"Test set size: {test.count():,}")

Models loaded and predictions generated successfully
Test set size: 171,667


In [0]:
#Confusion Matrices' 

from pyspark.sql.functions import count

def plot_confusion_matrix(preds, model_name):
    print(f"{model_name} Confusion Matrix ")
    cm = preds.groupBy("label", "prediction").count().orderBy("label", "prediction")
    display(cm)

plot_confusion_matrix(rf_preds, "Random Forest")
plot_confusion_matrix(dt_preds, "Decision Tree")
plot_confusion_matrix(lr_preds, "Logistic Regression")
plot_confusion_matrix(gbt_preds, "GBT (binary)")


Random Forest Confusion Matrix 


label,prediction,count
0.0,0.0,155838
0.0,2.0,9
1.0,0.0,12
1.0,1.0,9808
1.0,2.0,1
2.0,0.0,1
2.0,2.0,5981
3.0,0.0,4
3.0,3.0,13


Decision Tree Confusion Matrix 


label,prediction,count
0.0,0.0,155640
0.0,1.0,2
0.0,2.0,196
0.0,3.0,9
1.0,0.0,3
1.0,1.0,9808
1.0,2.0,10
2.0,0.0,1
2.0,2.0,5981
3.0,0.0,3


Logistic Regression Confusion Matrix 


label,prediction,count
0.0,0.0,155189
0.0,1.0,569
0.0,2.0,89
1.0,0.0,1592
1.0,1.0,8229
2.0,0.0,1757
2.0,1.0,2829
2.0,2.0,1396
3.0,0.0,16
3.0,1.0,1


GBT (binary) Confusion Matrix 


label,prediction,count
0.0,0.0,15802
0.0,1.0,18
1.0,0.0,55
1.0,1.0,155792


In [0]:
#Per Class Metrics

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

models = [
    ("Random Forest",       rf_preds),
    ("Decision Tree",       dt_preds),
    ("Logistic Regression", lr_preds),
]

print(f"{'Model':<25} {'Accuracy':<12} {'F1':<12} {'Precision':<12} {'Recall':<12}")
print("-" * 70)

for name, preds in models:
    accuracy  = evaluator.evaluate(preds, {evaluator.metricName: "accuracy"})
    f1        = evaluator.evaluate(preds, {evaluator.metricName: "f1"})
    precision = evaluator.evaluate(preds, {evaluator.metricName: "weightedPrecision"})
    recall    = evaluator.evaluate(preds, {evaluator.metricName: "weightedRecall"})
    print(f"{name:<25} {accuracy:<12.4f} {f1:<12.4f} {precision:<12.4f} {recall:<12.4f}")

# GBT separately with binary evaluator
binary_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
gbt_accuracy  = binary_evaluator.evaluate(gbt_preds, {binary_evaluator.metricName: "accuracy"})
gbt_f1        = binary_evaluator.evaluate(gbt_preds, {binary_evaluator.metricName: "f1"})
gbt_precision = binary_evaluator.evaluate(gbt_preds, {binary_evaluator.metricName: "weightedPrecision"})
gbt_recall    = binary_evaluator.evaluate(gbt_preds, {binary_evaluator.metricName: "weightedRecall"})
print(f"{'GBT (binary)':<25} {gbt_accuracy:<12.4f} {gbt_f1:<12.4f} {gbt_precision:<12.4f} {gbt_recall:<12.4f}")


Model                     Accuracy     F1           Precision    Recall      
----------------------------------------------------------------------
Random Forest             0.9998       0.9998       0.9998       0.9998      
Decision Tree             0.9987       0.9987       0.9987       0.9987      
Logistic Regression       0.9601       0.9532       0.9618       0.9601      
GBT (binary)              0.9996       0.9996       0.9996       0.9996      


In [0]:
#Features and Sci-kit learn Baseline:

import pandas as pd

# Feature importance for Random Forest
feature_cols = ["basisOfRecord", "collectionCode", "continent", "country", "lifeStage", "sex", "subDepartment", "order", "family", "phylum", "typeStatus", "month", "year", "individualCount"]

importances = rf_model.featureImportances
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": [importances[i] for i in range(len(feature_cols))]
}).sort_values("importance", ascending=False)

print("Feature Importances (Random Forest):")
print(importance_df.to_string(index=False))

# Scikit-learn baseline on small sample
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.tree import DecisionTreeClassifier as SklearnDT
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import accuracy_score
import numpy as np

# Take small sample for sklearn (single node)
sample = df.sample(fraction=0.01, seed=42).toPandas()
X = sample[["label"]].values  
y = sample["label"].values

# Use feature vector — convert sparse vector to array
from pyspark.ml.functions import vector_to_array
sample_spark = df.sample(fraction=0.01, seed=42)
sample_spark = sample_spark.withColumn("features_array", vector_to_array("features"))
sample_pd = sample_spark.toPandas()

X = np.array(sample_pd["features_array"].tolist())
y = sample_pd["label"].values

# Split
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Train sklearn models
sklearn_rf = SklearnRF(n_estimators=50, max_depth=10, random_state=42).fit(X_train, y_train)
sklearn_dt = SklearnDT(max_depth=10, random_state=42).fit(X_train, y_train)
sklearn_lr = SklearnLR(max_iter=100, random_state=42).fit(X_train, y_train)

print("\nScikit-learn Baseline (1% sample, single node):")
print(f"Sklearn Random Forest Accuracy: {accuracy_score(y_test, sklearn_rf.predict(X_test)):.4f}")
print(f"Sklearn Decision Tree Accuracy: {accuracy_score(y_test, sklearn_dt.predict(X_test)):.4f}")
print(f"Sklearn Logistic Regression Accuracy: {accuracy_score(y_test, sklearn_lr.predict(X_test)):.4f}")

Feature Importances (Random Forest):
        feature  importance
  subDepartment    0.370981
         phylum    0.210141
          order    0.193041
         family    0.085091
  basisOfRecord    0.041654
 collectionCode    0.033732
      continent    0.022865
        country    0.019333
           year    0.008757
individualCount    0.005588
            sex    0.003617
     typeStatus    0.003436
          month    0.001493
      lifeStage    0.000272

Scikit-learn Baseline (1% sample, single node):
Sklearn Random Forest Accuracy: 0.9670
Sklearn Decision Tree Accuracy: 0.9784
Sklearn Logistic Regression Accuracy: 0.6715


/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [0]:
#Scalability Analysis and Summary 
 
import time

# Scalability analysis - measure training time on different data sizes
fractions = [0.25, 0.5, 0.75, 1.0]
times = []

for fraction in fractions:
    sample = df.sample(fraction=fraction, seed=42)
    start = time.time()
    rf_temp = RandomForestClassificationModel.load(model_path + "random_forest")
    rf_temp.transform(sample).count()
    end = time.time()
    times.append((fraction, sample.count(), round(end - start, 2)))
    print(f"Fraction {fraction}: {sample.count():,} rows, {round(end-start,2)}s")

# Final summary table
results = spark.createDataFrame([
    ("Random Forest",       0.9998, 0.9998, "MLlib"),
    ("Decision Tree",       0.9987, 0.9987, "MLlib"),
    ("Logistic Regression", 0.9601, 0.9532, "MLlib"),
    ("GBT (binary)",        0.9996, 0.9996, "MLlib"),
    ("Tuned Random Forest", 0.9998, 0.9998, "MLlib"),
], ["Model", "Accuracy", "F1 Score", "Framework"])

display(results.orderBy("Accuracy", ascending=False))
print("all evaluation metrics generated")

Fraction 0.25: 285,549 rows, 5.26s
Fraction 0.5: 571,477 rows, 14.72s
Fraction 0.75: 857,680 rows, 14.16s
Fraction 1.0: 1,143,353 rows, 14.41s


Model,Accuracy,F1 Score,Framework
Random Forest,0.9998,0.9998,MLlib
Tuned Random Forest,0.9998,0.9998,MLlib
GBT (binary),0.9996,0.9996,MLlib
Decision Tree,0.9987,0.9987,MLlib
Logistic Regression,0.9601,0.9532,MLlib


all evaluation metrics generated


In [0]:
#ROC CURVE

from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col
import pandas as pd

# ROC data for Random Forest (binary - Animalia vs rest)
rf_binary_preds = rf_preds.withColumn("label", 
    when(col("label") == 0.0, 1.0).otherwise(0.0))

# Extract probability scores
from pyspark.ml.functions import vector_to_array

rf_roc = rf_binary_preds.withColumn("prob", 
    vector_to_array("probability")[0])

# Sample for export
roc_sample = rf_roc.select("label", "prob").sample(fraction=0.1, seed=42)
roc_pd = roc_sample.toPandas()

# Calculate ROC curve points
from sklearn.metrics import roc_curve, auc
fpr, tpr, thresholds = roc_curve(roc_pd["label"], roc_pd["prob"])
roc_auc = auc(fpr, tpr)

roc_df = pd.DataFrame({
    "False_Positive_Rate": fpr,
    "True_Positive_Rate": tpr,
})

roc_df.to_csv("/Volumes/workspace/default/museum/tableau/roc_curve.csv", index=False)
print(f"ROC AUC Score: {roc_auc:.4f}")
print("ROC curve data exported successfully")

ROC AUC Score: 1.0000
ROC curve data exported successfully
